# ForMoSA Statistical Tests and Model Selection

This notebook covers post-fit statistical diagnostics for ForMoSA nested-sampling runs:

1. Nested-sampling convergence diagnostics (logZ, effective sample size)
2. Goodness of fit (χ², reduced χ²)
3. Information criteria (AIC, BIC)
4. Bayesian model comparison (Bayes factors from logZ)
5. Practical use cases (fixed vs free parameter; different atmospheric grids)

**Prerequisites**: at least one (and for model comparison, two) completed ForMoSA fits.

---

## On autocorrelation, briefly

Many MCMC tutorials use chain autocorrelation length as a convergence diagnostic. **Don't do this with nested sampling.** NS samples aren't a Markov chain in the relevant sense — they're nested-shell draws weighted by their importance weights. Computing autocorrelation on NS samples is conceptually wrong and the number it produces is not meaningful for NS convergence. Use NS-appropriate diagnostics: `logZ` uncertainty, effective sample size, and re-runs with different seeds.


---
## 1. NS convergence diagnostics

### 1.1 Setup — load results

In [ ]:
import json
import numpy as np
from ForMoSA.nested_sampling.results import NSResults

# >>> REPLACE
results_json = 'PATH/TO/result_path/NS_results/results_pymultinest.json'

with open(results_json) as f:
    data = json.load(f)
results = NSResults.from_dict(data)

### 1.2 Evidence and its uncertainty

`results.logz` is a list `[logZ, logZ_err]` produced directly by the algorithm:

In [ ]:
logZ, logZ_err = results.logz
print(f'log Z = {logZ:.3f} +/- {logZ_err:.3f}')

**Rule of thumb**: `logZ_err ≲ 0.5` is typical for a well-converged PyMultiNest run with default `npoints`. If it's substantially larger, re-run with more live points.

### 1.3 Effective sample size (ESS)

The posterior is a *weighted* sample. A handful of high-weight points can dominate even when you have 10⁴ raw samples. The standard correction:

$$\mathrm{ESS} \;=\; \frac{1}{\sum_i \tilde w_i^2}$$

where $\tilde w_i$ are the normalized weights.

In [ ]:
w      = results.weights[results.burn_in:]
w_norm = w / w.sum()

ess        = 1.0 / np.sum(w_norm**2)
efficiency = ess / len(w)

print(f'raw samples = {len(w)}')
print(f'ESS         = {ess:.0f}')
print(f'efficiency  = {efficiency:.2%}')

**Interpretation**:

- ESS > 1000 → posterior quantiles are well-resolved
- ESS in 100–1000 → quantiles are noisy at the tails (2σ), medians are usually fine
- ESS < 100 → re-run with more live points; your posterior is dominated by a few samples

### 1.4 Re-running with a different seed

The most honest convergence check: run NS twice with different RNG seeds and compare `logZ` and the posterior medians. If they differ by more than `logZ_err`, you're under-converged. There's no programmatic shortcut; just run the fit twice.


---
## 2. Goodness of fit

### 2.1 Reduced χ²

> **Caveat**: χ² is only meaningful for likelihoods of the form `chi2` or `chi2_noisescaling`. For CCF-based likelihoods (e.g. HiRISE in MOSAIC mode), residuals don't reduce to χ² in the standard sense — skip this section for those datasets, or restrict the sum to your χ²-likelihood observations only.

In [ ]:
# Requires a full Analysis with fitted=True and ns_analysis available.
# >>> REPLACE with the appropriate loading code (see plotting tutorial, Section 2.3)
# from ForMoSA.analysis import Analysis
# analysis = Analysis(..., fitted=True, adapted=True)

# Per-observation χ²
chi2_total = 0.0
ndata      = 0

for i, obs in enumerate(analysis.ns.restricted_observations):
    model_flux = ns_analysis.best_fit[i].flux        # or .total_flux for high-contrast
    residuals  = (obs.flux - model_flux) / obs.err
    chi2_i     = np.sum(residuals**2)
    n_i        = len(obs.flux)

    chi2_total += chi2_i
    ndata      += n_i

    print(f'{obs.name:20s}  chi2 = {chi2_i:10.2f}   N = {n_i:5d}   chi2/N = {chi2_i/n_i:.3f}')

n_free = len(analysis.ns.results.free_parameters)
dof    = ndata - n_free
chi2_red = chi2_total / dof

print()
print(f'Total chi2 = {chi2_total:.2f}')
print(f'DOF        = {dof}')
print(f'chi2_red   = {chi2_red:.3f}')

**Interpretation**:

- `chi2_red ≈ 1` — fit consistent with stated errors
- `chi2_red >> 1` — model underfits, or errors are underestimated
- `chi2_red << 1` — errors are likely overestimated, or model is overfitting

Per-observation χ²/N is useful for diagnosing *which* dataset drives a poor fit in a joint inversion.


---
## 3. Information criteria

### 3.1 AIC and BIC

$$\mathrm{AIC} = 2k - 2\,\ln \hat L$$
$$\mathrm{BIC} = k \ln n - 2\,\ln \hat L$$

where $k$ is the number of free parameters, $n$ is the number of data points, and $\hat L$ is the maximum likelihood. Lower is better for both. AIC penalizes complexity weakly; BIC penalizes it strongly when `n` is large.

In [ ]:
# Best log-likelihood from posterior (after burn-in)
best_logL = results.logl[results.burn_in:].max()

k = len(results.free_parameters)
n = ndata                                # from section 2.1

AIC = 2*k - 2*best_logL
BIC = k*np.log(n) - 2*best_logL

print(f'best logL = {best_logL:.3f}')
print(f'k = {k}, n = {n}')
print(f'AIC = {AIC:.2f}')
print(f'BIC = {BIC:.2f}')

### 3.2 Comparing two fits

ΔAIC interpretation (Burnham & Anderson 2002, *Model Selection and Multimodel Inference*, 2nd ed., Springer):

| ΔAIC          | Support for the higher-AIC model    |
|---------------|-------------------------------------|
| 0–2           | substantial                          |
| 4–7           | considerably less                    |
| > 10          | essentially none                     |

A symmetric convention applies for ΔBIC, though some authors use slightly different bands.


---
## 4. Bayesian model comparison

### 4.1 Bayes factor from logZ

The Bayes factor between two models is the ratio of their marginal likelihoods (evidences). For NS, this falls straight out of `logZ`:

$$\ln B_{12} = \ln Z_1 - \ln Z_2$$

A positive value favors model 1.

In [ ]:
# >>> Load two fits' NSResults
# results_1 = NSResults.from_dict(json.load(open('PATH/model_1/NS_results/results_pymultinest.json')))
# results_2 = NSResults.from_dict(json.load(open('PATH/model_2/NS_results/results_pymultinest.json')))

# logB = results_1.logz[0] - results_2.logz[0]
# logB_err = np.hypot(results_1.logz[1], results_2.logz[1])
# print(f'log B_12 = {logB:.2f} +/- {logB_err:.2f}')

### 4.2 Interpretation

The convention most commonly cited is Kass & Raftery (1995), *JASA* 90, 773 — itself adapted from Jeffreys (1961), *Theory of Probability*, 3rd ed.

| `2 ln B_12`   | `ln B_12`     | Evidence in favor of model 1            |
|---------------|---------------|-----------------------------------------|
| 0 – 2         | 0 – 1         | not worth more than a bare mention      |
| 2 – 6         | 1 – 3         | positive                                |
| 6 – 10        | 3 – 5         | strong                                  |
| > 10          | > 5           | very strong                             |

Sources to cite (in your paper) depending on which convention you adopt:

- Kass & Raftery (1995), *JASA* 90, 773. DOI: 10.1080/01621459.1995.10476572
- Jeffreys (1961), *Theory of Probability*, Oxford University Press, 3rd ed.
- Trotta (2008), *Contemp. Phys.* 49, 71 — astronomy-focused, slightly different bands. DOI: 10.1080/00107510802066753

> The exact thresholds vary across sources. Pick one convention and cite it.

### 4.3 Bayes factor is prior-dependent — sanity-check it

`logZ` is the integral of the likelihood over the prior. If two models have very different prior volumes (e.g. different parameter ranges, or a model with extra unconstrained parameters), the Bayes factor can be dominated by prior shape rather than the data's preference.

Always sanity-check Bayes factors against χ² and information criteria. If `logB_12` says "model 1 strongly preferred" but `chi2_red` and AIC say the two are nearly identical, the Bayes factor is reflecting an *Occam penalty* from the unused volume in model 2's prior — not a real fit-quality difference. That's mathematically correct but worth being explicit about.


---
## 5. Practical use cases

### 5.1 Use case A — Fixed vs free `logg`

Run two ForMoSA fits with the same data and same other parameters; in one, fix `logg` to a literature value; in the other, let it float.

In [ ]:
# >>> REPLACE paths
results_fixed = NSResults.from_dict(json.load(open('PATH/fixed_logg/NS_results/results_pymultinest.json')))
results_free  = NSResults.from_dict(json.load(open('PATH/free_logg/NS_results/results_pymultinest.json')))

# Bayes factor in favor of "free logg"
logB = results_free.logz[0] - results_fixed.logz[0]
logB_err = np.hypot(results_free.logz[1], results_fixed.logz[1])
print(f'log B(free vs fixed) = {logB:.2f} +/- {logB_err:.2f}')

# Also report posterior on logg from the free fit
free_params = results_free.free_parameters
if 'logg' in free_params:
    j = free_params.index('logg')
    samples = results_free.samples[results_free.burn_in:, j]
    weights = results_free.weights[results_free.burn_in:]
    med = np.average(samples, weights=weights)
    lo  = np.percentile(samples, 16)   # un-weighted as a quick check
    hi  = np.percentile(samples, 84)
    print(f'logg (free fit, weighted mean): {med:.2f}')
    print(f'logg (un-weighted 16th–84th):   [{lo:.2f}, {hi:.2f}]')

**Reading the result**:

- `logB > 3` and the `logg` posterior is well-constrained (narrow, away from prior edges) → the data prefers a free `logg`; keep it free.
- `logB ≈ 0` and the `logg` posterior is broad (close to prior range) → data doesn't constrain `logg`; fixing it to a sensible literature value is justified, and the simpler model is favored on Occam grounds.
- `logB < -3` → fixing was strictly preferred; check the prior range you used on the free fit (a too-wide prior penalizes the free model unfairly).

### 5.2 Use case B — Different atmospheric grids

Run the same observations through two different grids (e.g. BT-Settl vs Sonora-Bobcat).

In [ ]:
# >>> REPLACE paths
results_grid_A = NSResults.from_dict(json.load(open('PATH/grid_A/NS_results/results_pymultinest.json')))
results_grid_B = NSResults.from_dict(json.load(open('PATH/grid_B/NS_results/results_pymultinest.json')))

logZ_A, errA = results_grid_A.logz
logZ_B, errB = results_grid_B.logz

print(f'log Z (grid A) = {logZ_A:.2f} +/- {errA:.2f}')
print(f'log Z (grid B) = {logZ_B:.2f} +/- {errB:.2f}')
print(f'log B_AB       = {logZ_A - logZ_B:.2f}  (+ favors A)')

**Important caveats for grid comparison**:

1. **Parameterization differences inflate Bayes factors**. If grid A uses `[M/H]` and grid B uses `[Fe/H]` with a different prior range, the prior-volume effect changes `logZ` even if the two grids fit equally well. For a clean comparison, use the *same* free parameter set with the *same* prior ranges across both grids.

2. **Wavelength coverage must be identical**. `logZ` scales with the number of data points. If you use different wavelength windows for the two fits, you can't compare them.

3. **When in doubt, also compare reduced χ²**. For atmospheric grids the typical difference is in *how well* the model reproduces line shapes — that shows up cleanly in χ², while it may be diluted in `logZ` by the prior volume.

### 5.3 Reporting

When reporting model selection in a paper, give **all four numbers**: `logZ`, `logZ_err`, `chi2_red`, and `AIC` (or `BIC`). They tell different stories and a referee will ask if you only give one.
